In [ ]:
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

G = nx.read_edgelist(
    "Wiki-Vote.txt",
    comments="#",
    nodetype=int,
    create_using=nx.DiGraph()
)

df = pd.read_csv("link_prediction_dataset.csv", header=None)
df.columns = ["u", "v", "label"]

positive = df[df.label == 1][["u", "v"]].values
negative = df[df.label == 0][["u", "v"]].values

H = G.copy()
H.remove_edges_from(positive)

model = Node2Vec(H, dimensions=16, walk_length=10, num_walks=50).fit()

# features
def emb(u, v):
    return model.wv[str(u)] * model.wv[str(v)]  # Hadamard

X = np.array([emb(u, v) for u, v in df[["u", "v"]].values])
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# model
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# metrics
pred = clf.predict(X_test)
print(classification_report(y_test, pred))